# Automatic Mixed Precision (AMP) Tutorial

## Overview

AMP uses FP16 for computation and FP32 for accumulation, providing ~2x speedup with minimal accuracy loss.

### Learning Objectives
- Understand FP16 vs FP32 trade-offs
- Implement AMP training with GradScaler
- Handle gradient overflow/underflow

### References
- Micikevicius et al., "Mixed Precision Training", ICLR 2018

## 1. Mathematical Foundation

### 1.1 Floating Point Formats

| Format | Sign | Exponent | Mantissa | Range | Precision |
|--------|------|----------|----------|-------|----------|
| FP32 | 1 | 8 | 23 | ±3.4e38 | ~7 digits |
| FP16 | 1 | 5 | 10 | ±65504 | ~3 digits |
| BF16 | 1 | 8 | 7 | ±3.4e38 | ~2 digits |

### 1.2 Loss Scaling

To prevent gradient underflow in FP16:

$$\tilde{L} = s \cdot L$$
$$\tilde{g} = s \cdot \nabla L$$
$$g = \tilde{g} / s$$

Where $s$ is the loss scale factor (typically 2^16 initially).

In [ ]:
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

## 2. Basic AMP Training

In [ ]:
def train_with_amp(model, dataloader, optimizer, criterion, device, epochs=5):
    """Training loop with Automatic Mixed Precision."""
    scaler = GradScaler()
    model.to(device)
    
    for epoch in range(epochs):
        total_loss = 0
        for data, target in dataloader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            
            # Forward pass with autocast
            with autocast():
                output = model(data)
                loss = criterion(output, target)
            
            # Backward pass with scaling
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

## 3. Dynamic Loss Scaling

In [ ]:
class DynamicLossScaler:
    """Dynamic loss scaling for mixed precision training."""
    
    def __init__(self, init_scale=2**16, growth_factor=2.0, 
                 backoff_factor=0.5, growth_interval=2000):
        self.scale = init_scale
        self.growth_factor = growth_factor
        self.backoff_factor = backoff_factor
        self.growth_interval = growth_interval
        self.steps_since_growth = 0
    
    def update(self, overflow: bool):
        if overflow:
            self.scale *= self.backoff_factor
            self.steps_since_growth = 0
        else:
            self.steps_since_growth += 1
            if self.steps_since_growth >= self.growth_interval:
                self.scale *= self.growth_factor
                self.steps_since_growth = 0
        return self.scale

## 4. Summary

### Key Takeaways

1. **AMP Benefits**: ~2x speedup, ~50% memory reduction
2. **Loss Scaling**: Prevents gradient underflow in FP16
3. **autocast**: Automatically selects precision per operation
4. **GradScaler**: Handles dynamic loss scaling

### Best Practices

- Use BF16 on Ampere+ GPUs (no scaling needed)
- Keep batch norm and softmax in FP32
- Monitor for NaN/Inf during training